<h2 align="center"><b>Simulating a bouncing ball</b></h2>
<div align=center>
<table style="width: 100%; border: none; border-collapse: collapse;">
<tr>

<style>
.emines-logo-wrap {
    position: relative;
    display: inline-block;
}

.emines-logo-wrap .emines-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.emines-logo-wrap .emines-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/emines_logo.svg');
    mask-image: url('../workshop_utils/assets/emines_logo.svg');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}
@media (prefers-color-scheme: dark) {
    .emines-logo-wrap .emines-logo-fill { background-color: #ffffff; }
}
.emines-logo-wrap .emines-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<!-- empty cell for centering purposes -->
<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Left side -->
<td style="width: 15%; border: none; text-align: right; vertical-align: middle;">
<div style="padding-right: 0;" class="emines-logo-wrap">
    <img class="emines-logo-sizer" src="../workshop_utils/assets/emines_logo.svg" alt="" aria-hidden="true">
    <div class="emines-logo-fill" role="img" aria-label="EMINES logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Video -->
<td style="width: 20%; border: none; text-align: center; vertical-align: middle;">
<video
src="../workshop_utils/assets/bouncing_ball_demo.mp4"
autoplay
muted
loop
playsinline
width="600">
</video>
</td>

<!-- Video -->
<td style="width: 20%; border: none; text-align: center; vertical-align: middle;">
<video
src="../workshop_utils/assets/bouncing_ball_gravity_demo.mp4"
autoplay
muted
loop
playsinline
width="600">
</video>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<style>
.ephysics-logo-wrap {
    position: relative;
    display: inline-block;
}

.ephysics-logo-wrap .ephysics-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.ephysics-logo-wrap .ephysics-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}

@media (prefers-color-scheme: dark) {
    .ephysics-logo-wrap .ephysics-logo-fill { background-color: #ffffff; }
}

.ephysics-logo-wrap .ephysics-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<!-- Right side -->
<td style="width: 15%; border: none; text-align: left; vertical-align: middle;">
<div style="padding-left: 0;" class="ephysics-logo-wrap">
    <img class="ephysics-logo-sizer" src="../workshop_utils/assets/ephysics_logo.png" alt="" aria-hidden="true">
    <div class="ephysics-logo-fill" role="img" aria-label="EPhy logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

</tr>
</table>
</div>

Let's apply the "general recipe" you just saw using a simple yet concrete example:<br><i>a bouncing ball that loses a small amount of energy with each jump.</i>

In [13]:
import pygame

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Defining the system — State variables — Parameters</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

Our physical system is a ball inside a rectangular box. Its state is its position and velocity:
$$\mathcal S = (x, y, v_x, v_y).$$

We use pixels as our distance unit and seconds as our time unit.\
`dt` is the time step: the amount of simulated time represented by one frame.

***


In [14]:
width, height = 1000, 600
screen_resolution = (width, height)
fps = 60
dt = 1.0 / fps

radius = 40
red = (255, 0, 0)
black = (0, 0, 0)

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Two cases — Constant velocity and gravity</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

With `gravity=False`, there is no acceleration:
$$\frac{d\vec v}{dt}=0.$$

With `gravity=True`, Newton's second law gives, in screen coordinates where positive `y` points downward:
$$a_x=0,\qquad a_y=g.$$

At each wall, the bounced velocity is multiplied by the coefficient of `restitution`, where `1` means no energy loss.

In both cases, the box walls are our boundary constraints. The same `advance` function handles both cases using the boolean `gravity` argument.

***



In [15]:
g = 980.0
restitution = 0.9

initial_x = width/2
initial_y = radius

initial_vx = 200.0
initial_vy = 100.0

x_min = radius ##
x_max = width - radius ##

y_min = radius ##
y_max = height - radius ##

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Discretization — Evolving the system</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

The acceleration is the second derivative of position, so each time step needs two Euler updates:
$$\vec v_{n+1}=\vec v_n+\vec a_n dt,\qquad \vec r_{n+1}=\vec r_n+\vec v_{n+1}dt.$$

Updating velocity before position is called <b>semi-implicit Euler</b>.
it behaves better for this bouncing system than using the old velocity for position.

***


In [ ]:
def advance(x, y, vx, vy, gravity):
    if not gravity:
        x += vx * dt   ## Euler position update
        y += vy * dt   ## Euler position update

    if gravity:
        ax = 0.0
        ay = g

        vx += ax * dt  ## Euler velocity update
        vy += ay * dt  ## Euler velocity update

        x += vx * dt   ## Euler position update
        y += vy * dt   ## Euler position update    


    if x < x_min or x > x_max:

        vx = -restitution * vx  ## wall bounce

        x = max(x_min, min(x, x_max))

    if y < y_min or y > y_max:

        vy = -restitution * vy  ## wall bounce
        
        y = max(y_min, min(y, y_max))

    return x, y, vx, vy


<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Visualizing the simulation</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

One GUI function runs either case in a separate pygame window. It passes the chosen `gravity` value to `advance`, draws the ball, and exits when the window is closed.
Run the calls below to compare the two cases, closing each window before continuing.

***



In [17]:
def run_demo(gravity):
    pygame.init()
    screen = pygame.display.set_mode(screen_resolution)
    case = "gravity" if gravity else "constant velocity"
    pygame.display.set_caption(f"E-Physics Workshop — Bouncing ball: {case}")
    clock = pygame.time.Clock()
    running = True

    x, y = initial_x, initial_y
    vx, vy = initial_vx, initial_vy

    try:
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
            if not running:
                break

            x, y, vx, vy = advance(x, y, vx, vy, gravity=gravity)

            screen.fill(black)
            pygame.draw.circle(screen, red, (int(x), int(y)), radius)
            pygame.display.flip()
            clock.tick(fps)
    finally:
        pygame.quit()


In [18]:
run_demo(gravity=False)


In [19]:
run_demo(gravity=True)
